In [0]:
from pyspark.sql.functions import col, when, timestamp_diff

In [0]:
"""
read in bronze layer (raw) data
"""
df = spark.read.table('workspace.01_bronze.yellow_trips_raw')
display(df.limit(10))

In [0]:
"""
filter on pickup times, getting times betwween jan (inclusive) and july (exclusive)
this ensures we are only getting those 6 months of data.  Any outlier data or data in there by mistake is filtered out.
NOTE:  this is a good practice.  Never assume what you downloaded is correct
"""
df = df.filter("tpep_pickup_datetime >= '2025-01-01' AND tpep_pickup_datetime < '2025-07-01'")

In [0]:
"""
Transformation
NOTE: conditional operations using 'when(col)'
1. replace VendorID with vendor's name, and alias the column to 'vendor'
2. get the trip duration in minutes by subtract pickup from dropoff
3. replace RatecodeID with rate name, and alias the column to 'rate_type'
4. alias pickup and dropoff columns
5. replace payment_type codes with payment names

"""
df = df.select(
    when(col('VendorID') == 1, "Creative Mobile Technologies, LLC") \
        .when(col('VendorID') == 2, "Curb Mobility, LLC") \
        .when(col('VendorID') == 6, "Myle Technologies Inc") \
        .when(col('VendorID') == 7, "Helix") \
        .otherwise("Unknown") \
        .alias('vendor'),
               
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    timestamp_diff('MINUTE', df.tpep_pickup_datetime, df.tpep_dropoff_datetime) \
        .alias("trip_duration"),          
    "passenger_count",
    "trip_distance",

    when(col('RatecodeID') == 1, "Standard Rate") \
        .when(col('RatecodeID') == 2, "JFK") \
        .when(col('RatecodeID') == 3, "Newark") \
        .when(col('RatecodeID') == 4, "Nassau or WestChester") \
        .when(col('RatecodeID') == 5, "Negotiated Fare") \
        .when(col('RatecodeID') == 6, "Group Ride") \
        .otherwise("Unknown")
        .alias("rate_type"),
    
    "store_and_fwd_flag",
    col('PULocationID').alias('pu_location_id'),
    col('DOLocationID').alias('do_location_id'),

    when(col('payment_type') == 0, "Flex Fare trip") \
        .when(col('payment_type') == 1, "Credit card") \
        .when(col('payment_type') == 2, "Cash") \
        .when(col('payment_type') == 3, "No charge") \
        .when(col('payment_type') == 4, "Dispute") \
        .when(col('payment_type') == 0, "Voided trip") \
        .otherwise("Unknown") \
        .alias('payment_type'),

    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    col("Airport_fee").alias("airport_fee"),
    "cbd_congestion_fee",
    "processed_timestamp"
)

In [0]:
"""
verify dataframe
"""
display(df.limit(10))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS 02_silver.yellow_trips_cleansed")

In [0]:
"""
write cleaned data to table
"""
df.write.saveAsTable('02_silver.yellow_trips_cleansed')

In [0]:
"""
veify table
"""
spark.read.table('02_silver.yellow_trips_cleansed').display()